# Laboratorio 4 — Análisis de Datos GeoEspaciales
### CC3084 — Data Science | UVG | Semestre II - 2026
**Integrantes:** Jose Ordoñez - 231329, Adrián González - 23152, José Antón - 221041

Link de GitHub: https://github.com/Ikeel04/Labs-DS.git


## Ejercicio 1. Conexión con el API de Sentinel-2 (openEO)

Usamos el backend openEO de la **Copernicus Data Space Ecosystem (CDSE)**, que es el
punto de acceso oficial (gratuito, con cuenta Copernicus) a las colecciones de
Sentinel-2 L2A. La autenticación es OIDC (abre una ventana/URL de login la primera vez
y luego cachea el token localmente).

In [2]:
# Si no lo tienes instalado:
# %pip install openeo geopandas shapely matplotlib pandas --quiet

import openeo
from pathlib import Path
import pandas as pd

# --- Conexión al backend openEO de Copernicus Data Space Ecosystem ---
OPENEO_URL = "https://openeo.dataspace.copernicus.eu"

connection = openeo.connect(OPENEO_URL)

# Autenticación OIDC: la primera vez abrirá una URL/navegador para iniciar sesión
# con tu usuario y contraseña de Copernicus. El token queda cacheado para las
# siguientes ejecuciones (no es necesario volver a loguearse cada vez).
connection.authenticate_oidc()

print("Conectado a:", OPENEO_URL)
print("Usuario autenticado correctamente.")

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=GSXV-MLTI 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.
Conectado a: https://openeo.dataspace.copernicus.eu
Usuario autenticado correctamente.


In [3]:
# Verificamos que la colección Sentinel-2 L2A esté disponible en el backend
colecciones = connection.list_collection_ids()
assert "SENTINEL2_L2A" in colecciones, "La colección SENTINEL2_L2A no está disponible en este backend"
print("Colección SENTINEL2_L2A disponible. Listo para continuar.")

Colección SENTINEL2_L2A disponible. Listo para continuar.


## Ejercicio 2. Obtención de los datos raster necesarios

Para reducir tiempos de descarga y almacenamiento, **solo se descargan las bandas
necesarias para calcular NDVI y NDWI**:

- **NDVI:** `B04` (rojo) y `B08` (infrarrojo cercano)
- **NDWI:** `B03` (verde) y `B08` (infrarrojo cercano)

Es decir, con las tres bandas `B03`, `B04`, `B08` cubrimos ambos índices en una sola
descarga por imagen (evitamos descargar la escena completa).

Usamos exclusivamente:
- Las coordenadas (bounding box) de cada lago dadas en el enunciado.
- Las fechas oficiales de la tabla del laboratorio (11 fechas por lago).

In [4]:
# --- Coordenadas de cada lago (dadas en el enunciado del laboratorio) ---
lagos_bbox = {
    "atitlan": {
        "west": -91.326256,
        "east": -91.07151,
        "south": 14.5948,
        "north": 14.750979,
    },
    "amatitlan": {
        "west": -90.638065,
        "east": -90.512924,
        "south": 14.412347,
        "north": 14.493799,
    },
}

# --- Fechas oficiales por lago (tal como las proporciona el laboratorio) ---
fechas_atitlan = [
    "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17", "2025-11-21",
    "2025-12-29", "2026-02-12", "2026-03-24", "2026-04-13", "2026-04-28",
    "2026-07-22",
]

fechas_amatitlan = [
    "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24", "2026-01-08",
    "2026-02-02", "2026-02-07", "2026-03-29", "2026-04-13", "2026-04-28",
    "2026-06-19",
]

fechas_por_lago = {
    "atitlan": fechas_atitlan,
    "amatitlan": fechas_amatitlan,
}

for lago, fechas in fechas_por_lago.items():
    print(f"{lago}: {len(fechas)} fechas oficiales")

atitlan: 11 fechas oficiales
amatitlan: 11 fechas oficiales


In [5]:
# --- Bandas mínimas necesarias: B03 (verde), B04 (rojo), B08 (NIR) ---
BANDAS = ["B03", "B04", "B08"]

# Carpeta de salida para los raster descargados
OUTPUT_DIR = Path("data/raster")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def siguiente_dia(fecha_str: str) -> str:
    """Devuelve la fecha siguiente en formato YYYY-MM-DD (para construir un
    rango temporal de un solo día, requerido por openEO)."""
    fecha = pd.to_datetime(fecha_str)
    return (fecha + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


def descargar_imagen(lago: str, fecha: str, bbox: dict, bandas=BANDAS,
                      max_cloud_cover: float = 20.0) -> Path:
    """Descarga únicamente las bandas necesarias para una fecha y un lago
    dados, usando el bounding box provisto. Guarda el resultado como GeoTIFF.
    """
    destino = OUTPUT_DIR / lago / f"{lago}_{fecha}.tif"
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.exists():
        print(f"[omitido] Ya existe: {destino}")
        return destino

    datacube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[fecha, siguiente_dia(fecha)],
        bands=bandas,
        max_cloud_cover=max_cloud_cover,
    )

    # Nos quedamos con la mediana temporal por si hay más de una escena en
    # el rango (debería ser una sola, dado que el rango es de un día)
    datacube = datacube.reduce_dimension(dimension="t", reducer="median")

    print(f"Descargando {lago} - {fecha} ...")
    datacube.download(str(destino))
    print(f"  -> guardado en {destino}")
    return destino

In [ ]:
# --- Descarga de todas las imágenes (solo bandas necesarias) para ambos lagos ---
rutas_descargadas = []

for lago, bbox in lagos_bbox.items():
    for fecha in fechas_por_lago[lago]:
        ruta = descargar_imagen(lago, fecha, bbox)
        rutas_descargadas.append({"lago": lago, "fecha": fecha, "ruta": str(ruta)})

df_descargas = pd.DataFrame(rutas_descargadas)
df_descargas

Descargando atitlan - 2025-01-18 ...
  -> guardado en data\raster\atitlan\atitlan_2025-01-18.tif
Descargando atitlan - 2025-04-13 ...
  -> guardado en data\raster\atitlan\atitlan_2025-04-13.tif
Descargando atitlan - 2025-05-13 ...
  -> guardado en data\raster\atitlan\atitlan_2025-05-13.tif
Descargando atitlan - 2025-07-17 ...
  -> guardado en data\raster\atitlan\atitlan_2025-07-17.tif
Descargando atitlan - 2025-11-21 ...
  -> guardado en data\raster\atitlan\atitlan_2025-11-21.tif
Descargando atitlan - 2025-12-29 ...
  -> guardado en data\raster\atitlan\atitlan_2025-12-29.tif
Descargando atitlan - 2026-02-12 ...
  -> guardado en data\raster\atitlan\atitlan_2026-02-12.tif
Descargando atitlan - 2026-03-24 ...
  -> guardado en data\raster\atitlan\atitlan_2026-03-24.tif
Descargando atitlan - 2026-04-13 ...
  -> guardado en data\raster\atitlan\atitlan_2026-04-13.tif
Descargando atitlan - 2026-04-28 ...
  -> guardado en data\raster\atitlan\atitlan_2026-04-28.tif
Descargando atitlan - 2026-07-

In [ ]:
# Guardamos el índice de archivos descargados para usarlo en los siguientes
# ejercicios (cálculo de índices, análisis temporal y espacial)
df_descargas.to_csv("data/indice_descargas.csv", index=False)
print("Total de imágenes descargadas:", len(df_descargas))
df_descargas.groupby("lago").size()

## Ejercicio 3.

descarga solo B05 (rápido, ~22 imágenes de 1 banda)

In [ ]:
BANDA_EXTRA = "B05"

def descargar_banda_extra(lago, fecha, bbox, banda=BANDA_EXTRA, max_cloud_cover=20.0):
    destino = OUTPUT_DIR / lago / f"{lago}_{fecha}_{banda}.tif"
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"[omitido] Ya existe: {destino}")
        return destino
    datacube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[fecha, siguiente_dia(fecha)],
        bands=[banda],
        max_cloud_cover=max_cloud_cover,
    )
    datacube = datacube.reduce_dimension(dimension="t", reducer="median")
    print(f"Descargando {lago} - {fecha} - {banda} ...")
    datacube.download(str(destino))
    return destino

rutas_b05 = []
for lago, bbox in lagos_bbox.items():
    for fecha in fechas_por_lago[lago]:
        ruta = descargar_banda_extra(lago, fecha, bbox)
        rutas_b05.append({"lago": lago, "fecha": fecha, "ruta_b05": str(ruta)})

df_b05 = pd.DataFrame(rutas_b05)
df_descargas = df_descargas.merge(df_b05, on=["lago", "fecha"])

funciones para NDVI, NDWI y cianobacteria (NDCI/Chl-a)

In [ ]:
import rasterio
import numpy as np

def leer_bandas(ruta_principal, ruta_b05):
    with rasterio.open(ruta_principal) as src:
        b03 = src.read(1).astype("float32")
        b04 = src.read(2).astype("float32")
        b08 = src.read(3).astype("float32")
    with rasterio.open(ruta_b05) as src2:
        b05 = src2.read(1).astype("float32")
    return b03, b04, b08, b05

def calcular_indices(b03, b04, b08, b05):
    eps = 1e-6
    ndvi = (b08 - b04) / (b08 + b04 + eps)
    ndwi = (b03 - b08) / (b03 + b08 + eps)
    ndci = (b05 - b04) / (b05 + b04 + eps)
    # Modelo exponencial calibrado del script de cianobacteria de Sentinel Hub
    # (Kravitz & Matthews, 2020 — custom-scripts.sentinel-hub.com)
    chl_a = 17.441 * np.exp(4.7038 * ndci)
    return ndvi, ndwi, chl_a

calcular índices para todas las imágenes

In [ ]:
resultados = []
for _, fila in df_descargas.iterrows():
    b03, b04, b08, b05 = leer_bandas(fila["ruta"], fila["ruta_b05"])
    ndvi, ndwi, chl_a = calcular_indices(b03, b04, b08, b05)
    resultados.append({
        "lago": fila["lago"],
        "fecha": fila["fecha"],
        "ndvi_prom": np.nanmean(ndvi),
        "ndwi_prom": np.nanmean(ndwi),
        "cianobacteria_prom": np.nanmean(chl_a),
    })

df_indices = pd.DataFrame(resultados)
df_indices["fecha"] = pd.to_datetime(df_indices["fecha"])
df_indices = df_indices.sort_values(["lago", "fecha"])
df_indices.to_csv("data/indices_por_imagen.csv", index=False)
df_indices

mapa del índice de cianobacteria por lago (pide la rúbrica)

In [ ]:
import matplotlib.pyplot as plt

for lago in lagos_bbox:
    fila = df_descargas[df_descargas["lago"] == lago].iloc[0]
    b03, b04, b08, b05 = leer_bandas(fila["ruta"], fila["ruta_b05"])
    _, _, chl_a = calcular_indices(b03, b04, b08, b05)

    plt.figure(figsize=(6, 5))
    plt.imshow(chl_a, cmap="RdYlGn_r")
    plt.colorbar(label="Chl-a (cianobacteria)")
    plt.title(f"Índice de cianobacteria — {lago} ({fila['fecha']})")
    plt.axis("off")
    plt.show()

## Ejercicio 4: análisis temporal

In [ ]:
# 4.2 Evolución temporal por lago
plt.figure(figsize=(10, 5))
for lago in df_indices["lago"].unique():
    sub = df_indices[df_indices["lago"] == lago]
    plt.plot(sub["fecha"], sub["cianobacteria_prom"], marker="o", label=lago)

plt.xlabel("Fecha")
plt.ylabel("Índice promedio de cianobacteria (Chl-a)")
plt.title("Evolución temporal del índice de cianobacteria por lago")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("data/evolucion_temporal_cianobacteria.png")
plt.show()

# 4.3 Picos de floración por lago
picos = df_indices.loc[df_indices.groupby("lago")["cianobacteria_prom"].idxmax()]
picos[["lago", "fecha", "cianobacteria_prom"]]